[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdsanch1/simrc/blob/master/02.%20Parte%202/14.%20Clase%2014/14Class%20NB.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jdsanch1/SimRC/master?labpath=02.%20Parte%202%2F14.%20Clase%2014%2F14Class%20NB.ipynb)

In [ ]:
import importlib, subprocess, sys

_required = ["yfinance", "pandas", "numpy", "matplotlib", "seaborn", "scipy", "sklearn", "statsmodels"]
_missing  = [pkg for pkg in _required if importlib.util.find_spec(pkg) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _missing)

# Clase 14: Opciones barrera (knock-in / knock-out)

**[Juan Diego Sánchez Torres](https://www.researchgate.net/profile/Juan_Diego_Sanchez_Torres)**  
*Profesor*, [MAF ITESO](http://maf.iteso.mx/web/general/detalle?group_id=5858156)  
dsanchez@iteso.mx

## Objetivos de aprendizaje

- Entender las **opciones barrera** como derivados path-dependent.
- Clasificar los 4 tipos: up/down × in/out.
- Valuar opciones barrera por **simulación Monte Carlo** de trayectorias completas.
- Comparar precios de opciones barrera con opciones europeas estándar.
- Entender por qué las opciones barrera **rompen la convexidad** (Boyd §3.2).

---

## Introducción teórica

### ¿Qué son las opciones barrera?

A diferencia de las opciones europeas (Clase 3), cuyo payoff solo depende del precio final $S_T$, las **opciones barrera** dependen de **toda la trayectoria** del precio $\{S_t\}_{0 \leq t \leq T}$. Se activan o desactivan cuando el precio toca un nivel predefinido $B$ (la barrera).

### Tipos de opciones barrera

| Tipo | Barrera | Efecto | Ejemplo |
|------|---------|--------|---------|
| **Down-and-out** | $B < S_0$ | Se **desactiva** si $\min_t S_t \leq B$ | Protección que desaparece si hay crash |
| **Down-and-in** | $B < S_0$ | Se **activa** si $\min_t S_t \leq B$ | Seguro que solo se activa en crisis |
| **Up-and-out** | $B > S_0$ | Se **desactiva** si $\max_t S_t \geq B$ | Call con techo de ganancia |
| **Up-and-in** | $B > S_0$ | Se **activa** si $\max_t S_t \geq B$ | Call que requiere subida previa |

### Relación in + out = europeo

Para cada par (in, out) con la misma barrera, strike y vencimiento:

$$
V_{\text{in}} + V_{\text{out}} = V_{\text{europeo}}
$$

Esto se debe a que en toda trayectoria, **exactamente una** de las dos opciones está activa.

### Payoff formal

El payoff de una **down-and-out call** es:

$$
V_T = \max(S_T - K, 0) \cdot \mathbf{1}\!\left\{\min_{0 \leq t \leq T} S_t > B\right\}
$$

La función indicadora $\mathbf{1}\{\cdot\}$ **no es convexa**, lo que implica que las opciones barrera no se pueden optimizar con herramientas convexas estándar (Boyd §3.2). Por ello, la simulación Monte Carlo es el método principal de valuación.

## 1. Datos y parámetros

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

sns.set_theme(style="darkgrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 100, "figure.figsize": (12, 5)})
pd.set_option("display.precision", 4)

In [ ]:
ticker = "AAPL"
data = yf.download(ticker, start="2025-01-01", end="2025-03-27",
                   auto_adjust=True, progress=False)
closes = data["Close"].dropna()
daily_returns = np.log(closes / closes.shift(1)).dropna()

S0 = closes.iloc[-1]
mu = daily_returns.mean()
sigma = daily_returns.std()
r_f = 0.045 / 252  # Tasa libre de riesgo diaria

print(f"Precio actual S₀ = ${S0:.2f}")
print(f"μ diario = {mu:.6f}, σ diario = {sigma:.6f}")

---
## 2. Simulación de trayectorias completas

Para opciones barrera, necesitamos **toda la trayectoria**, no solo el precio final. Simulamos $N$ trayectorias bajo la medida risk-neutral:

$$
S_{t+1} = S_t \exp\!\left(\left(r_f - \frac{\sigma^2}{2}\right)\Delta t + \sigma\sqrt{\Delta t} \, Z_t\right), \qquad Z_t \sim \mathcal{N}(0,1)
$$

In [ ]:
np.random.seed(42)
T = 180          # Días al vencimiento
N = 50000        # Trayectorias
K = 170          # Strike
dt = 1           # Paso = 1 día

# Drift risk-neutral
drift = (r_f - 0.5 * sigma**2) * dt

# Simular trayectorias completas
Z = np.random.randn(T, N)
log_returns = drift + sigma * np.sqrt(dt) * Z
log_prices = np.log(S0) + np.cumsum(log_returns, axis=0)
paths = np.exp(log_prices)  # shape: (T, N)

# Insertar S0 al inicio
paths = np.vstack([np.full(N, S0), paths])  # shape: (T+1, N)

print(f"Trayectorias: {N:,} × {T+1} pasos")
print(f"Precio final — media: ${paths[-1].mean():.2f}, std: ${paths[-1].std():.2f}")

In [ ]:
# Visualizar 50 trayectorias
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(paths[:, :50], alpha=0.15, color="steelblue")
ax.axhline(K, color="red", linestyle="--", label=f"Strike K = {K}")
ax.axhline(S0, color="black", linestyle=":", label=f"S₀ = {S0:.0f}")
ax.set_xlabel("Días")
ax.set_ylabel("Precio (USD)")
ax.set_title(f"50 trayectorias simuladas ({ticker}, T={T} días)")
ax.legend()
plt.tight_layout()

---
## 3. Valuación de opciones europeas (referencia)

Primero calculamos el precio de la call y put europeas estándar como referencia:

In [ ]:
ST = paths[-1]  # Precios finales
discount = np.exp(-r_f * T)

call_euro = discount * np.maximum(ST - K, 0).mean()
put_euro = discount * np.maximum(K - ST, 0).mean()

print(f"Call europea: ${call_euro:.4f}")
print(f"Put europea:  ${put_euro:.4f}")

---
## 4. Opciones barrera: down-and-out / down-and-in

### Down-and-out call

Se **desactiva** si el precio toca la barrera inferior $B$ en algún momento. Solo paga si el precio nunca cayó por debajo de $B$.

In [ ]:
B_down = 150  # Barrera inferior (por debajo de S0)

# Mínimo a lo largo de cada trayectoria
path_min = paths.min(axis=0)

# Down-and-out: activa solo si min > B
survived_do = path_min > B_down
call_do = discount * (np.maximum(ST - K, 0) * survived_do).mean()

# Down-and-in: activa solo si min <= B
triggered_di = path_min <= B_down
call_di = discount * (np.maximum(ST - K, 0) * triggered_di).mean()

print(f"Barrera B = ${B_down}")
print(f"  Down-and-out call: ${call_do:.4f}")
print(f"  Down-and-in call:  ${call_di:.4f}")
print(f"  Suma (in + out):   ${call_do + call_di:.4f}")
print(f"  Call europea:      ${call_euro:.4f}")
print(f"  Verificación in + out ≈ europeo: {'✓' if abs(call_do + call_di - call_euro) < 0.01 else '✗'}")

---
## 5. Opciones barrera: up-and-out / up-and-in

### Up-and-out call

Se **desactiva** si el precio toca la barrera superior $B$. Paga solo si el precio nunca subió por encima de $B$.

In [ ]:
B_up = 250  # Barrera superior (por encima de S0)

# Máximo a lo largo de cada trayectoria
path_max = paths.max(axis=0)

# Up-and-out: activa solo si max < B
survived_uo = path_max < B_up
call_uo = discount * (np.maximum(ST - K, 0) * survived_uo).mean()

# Up-and-in: activa solo si max >= B
triggered_ui = path_max >= B_up
call_ui = discount * (np.maximum(ST - K, 0) * triggered_ui).mean()

print(f"Barrera B = ${B_up}")
print(f"  Up-and-out call: ${call_uo:.4f}")
print(f"  Up-and-in call:  ${call_ui:.4f}")
print(f"  Suma (in + out): ${call_uo + call_ui:.4f}")
print(f"  Call europea:    ${call_euro:.4f}")

---
## 6. Efecto de la barrera en el precio

Variamos el nivel de la barrera para ver cómo afecta al precio de la opción:

In [ ]:
barriers_down = np.linspace(100, S0 - 5, 30)
barriers_up = np.linspace(S0 + 5, 350, 30)

prices_do = [discount * (np.maximum(ST - K, 0) * (path_min > b)).mean() for b in barriers_down]
prices_uo = [discount * (np.maximum(ST - K, 0) * (path_max < b)).mean() for b in barriers_up]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(barriers_down, prices_do, "o-", markersize=3, color="tab:blue")
axes[0].axhline(call_euro, color="red", linestyle="--", label=f"Call europea = ${call_euro:.2f}")
axes[0].set_xlabel("Barrera B (down)")
axes[0].set_ylabel("Precio de la opción")
axes[0].set_title("Down-and-out call vs. nivel de barrera")
axes[0].legend()

axes[1].plot(barriers_up, prices_uo, "o-", markersize=3, color="tab:orange")
axes[1].axhline(call_euro, color="red", linestyle="--", label=f"Call europea = ${call_euro:.2f}")
axes[1].set_xlabel("Barrera B (up)")
axes[1].set_ylabel("Precio de la opción")
axes[1].set_title("Up-and-out call vs. nivel de barrera")
axes[1].legend()

plt.suptitle("Efecto de la barrera en el precio de la opción", fontsize=14)
plt.tight_layout()

### Interpretación

- **Down-and-out**: cuando $B \to 0$, la barrera nunca se toca y el precio converge a la call europea. Cuando $B \to S_0$, casi todas las trayectorias tocan la barrera y el precio se acerca a cero.
- **Up-and-out**: cuando $B \to \infty$, la barrera nunca se toca y el precio converge a la call europea. Cuando $B \to S_0$, casi todas las trayectorias la tocan y el precio se acerca a cero.
- Las opciones barrera son **más baratas** que las europeas (siempre $V_{\text{out}} \leq V_{\text{europeo}}$).

---
## 7. Visualización de trayectorias con barrera

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 50 trayectorias: down-and-out
for i in range(50):
    color = "green" if path_min[i] > B_down else "red"
    alpha = 0.4 if path_min[i] > B_down else 0.15
    axes[0].plot(paths[:, i], color=color, alpha=alpha, linewidth=0.5)
axes[0].axhline(B_down, color="red", linewidth=2, linestyle="--", label=f"Barrera B = {B_down}")
axes[0].axhline(K, color="blue", linewidth=1, linestyle=":", label=f"Strike K = {K}")
axes[0].set_title(f"Down-and-out call (B={B_down})")
axes[0].set_xlabel("Días")
axes[0].set_ylabel("Precio (USD)")
axes[0].legend(fontsize=9)

# 50 trayectorias: up-and-out
for i in range(50):
    color = "green" if path_max[i] < B_up else "red"
    alpha = 0.4 if path_max[i] < B_up else 0.15
    axes[1].plot(paths[:, i], color=color, alpha=alpha, linewidth=0.5)
axes[1].axhline(B_up, color="red", linewidth=2, linestyle="--", label=f"Barrera B = {B_up}")
axes[1].axhline(K, color="blue", linewidth=1, linestyle=":", label=f"Strike K = {K}")
axes[1].set_title(f"Up-and-out call (B={B_up})")
axes[1].set_xlabel("Días")
axes[1].set_ylabel("Precio (USD)")
axes[1].legend(fontsize=9)

plt.suptitle("Trayectorias: verde = sobrevive, rojo = toca barrera", fontsize=13)
plt.tight_layout()

---
## 8. Tabla resumen de precios

In [ ]:
summary = pd.DataFrame({
    "Tipo": ["Call europea", f"Down-and-out (B={B_down})", f"Down-and-in (B={B_down})",
             f"Up-and-out (B={B_up})", f"Up-and-in (B={B_up})"],
    "Precio": [call_euro, call_do, call_di, call_uo, call_ui],
    "% de europea": [100, call_do/call_euro*100, call_di/call_euro*100,
                     call_uo/call_euro*100, call_ui/call_euro*100],
    "Tray. activas (%)": [100, survived_do.mean()*100, triggered_di.mean()*100,
                           survived_uo.mean()*100, triggered_ui.mean()*100],
})
summary.set_index("Tipo")

---

## Navegación del curso

← **Anterior**: Clase 13: Portafolio con bono (MC vs Markowitz)  
→ **Siguiente**: Clase 15: Programación estocástica con Pyomo y CVXPY

---

## 9. Referencias bibliográficas

- **Boyd, S. & Vandenberghe, L.** (2004). *Convex Optimization*. Cambridge University Press. — §3.2 (preservación de convexidad: indicadoras la rompen).
- **Glasserman, P.** (2003). *Monte Carlo Methods in Financial Engineering*. Springer. — Cap. 6: Barrier options.
- **Hull, J. C.** (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson. — Cap. 26: Exotic Options.
- **Luenberger, D. G.** (2013). *Investment Science* (2nd ed.). Oxford University Press.
- **Tsay, R. S.** (2010). *Analysis of Financial Time Series* (3rd ed.). Wiley.
- **Venegas Martínez, F.** (2008). *Riesgos financieros y económicos* (2a ed.). Cengage Learning. — Cap. 8: Opciones exóticas.